# 01. Извлечение данных из Hive

**User Story 1**: Как аналитик, я хочу извлечь N-hop окружение seed-компании из Hive, чтобы получить данные для построения графа связей.

**Процесс**:
1. Задаём seed-компанию (client_uk) и параметры
2. Расширяем окружение через транзакции (hop by hop)
3. Извлекаем узлы, транзакционные, доверенные и зарплатные рёбра
4. Сохраняем в Parquet-файлы

**Предпосылки**: Запустите `00_verify_schema.ipynb` для проверки колонок.

---

## Параметры

In [ ]:
# =====================================================
# НАСТРОЙТЕ ЭТИ ПАРАМЕТРЫ ПЕРЕД ЗАПУСКОМ
# =====================================================

SEED_CLIENT_UK = 12345678      # Замените на реальный client_uk seed-компании
N_HOPS = 2                      # Количество хопов (1-3)
START_DATE = '2025-01-01'       # Начало периода
END_DATE = '2025-12-31'         # Конец периода

# Директория для Parquet-файлов (абсолютный путь на локальной ФС)
import os
OUTPUT_DIR = os.path.join(os.path.dirname(os.getcwd()), 'data')

## Инициализация

In [ ]:
import sys
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from pyspark.sql import SparkSession
from src import config
from src.etl import extract_seed_neighborhood

spark = SparkSession.builder \
    .enableHiveSupport() \
    .getOrCreate()

print(f"Spark version: {spark.version}")
print(f"Database: {config.HIVE_DATABASE}")
print(f"Seed: {SEED_CLIENT_UK}, Hops: {N_HOPS}, Period: {START_DATE} — {END_DATE}")

## Извлечение окружения

In [ ]:
paths = extract_seed_neighborhood(
    spark=spark,
    seed_client_uk=SEED_CLIENT_UK,
    n_hops=N_HOPS,
    start_date=START_DATE,
    end_date=END_DATE,
    output_dir=OUTPUT_DIR,
)

print("\nOutput files:")
for name, path in paths.items():
    print(f"  {name}: {path}")

## Сводная статистика

In [ ]:
import pandas as pd

# Load and show summary — используем file:// для Spark read с локальной ФС
spark_prefix = f'file://{os.path.abspath(OUTPUT_DIR)}'

nodes_df = spark.read.parquet(f'{spark_prefix}/nodes.parquet')
tx_df = spark.read.parquet(f'{spark_prefix}/transaction_edges.parquet')
auth_df = spark.read.parquet(f'{spark_prefix}/authority_edges.parquet')
sal_df = spark.read.parquet(f'{spark_prefix}/salary_edges.parquet')

print("=" * 50)
print("EXTRACTION SUMMARY")
print("=" * 50)
print(f"Nodes:              {nodes_df.count():,}")
print(f"Transaction edges:  {tx_df.count():,}")
print(f"Authority edges:    {auth_df.count():,}")
print(f"Salary edges:       {sal_df.count():,}")

In [ ]:
# Node distribution by type
print("\nNodes by type:")
nodes_df.groupBy('client_type_name').count().orderBy('count', ascending=False).show(truncate=False)

In [ ]:
# Node distribution by hop distance
hop_df = spark.read.parquet(f'{spark_prefix}/hop_distances.parquet')
print("Nodes by hop distance:")
hop_df.groupBy('hop_distance').count().orderBy('hop_distance').show()

In [ ]:
# Sample data previews
print("\n--- Nodes sample ---")
nodes_df.show(5, truncate=40)

print("\n--- Transaction edges sample ---")
tx_df.show(5, truncate=30)

print("\n--- Authority edges sample ---")
auth_df.show(5, truncate=30)

print("\n--- Salary edges sample ---")
sal_df.show(5, truncate=30)

---

**Следующий шаг**: Откройте `02_graph_construction.ipynb` для построения графа из извлечённых данных.